# 06. Что мы в итоге построили

Финальная короткая глава нужна, чтобы workshop не заканчивался последней code cell. Здесь фиксируем архитектуру, примитивы и честный scope.


![Final architecture](../visuals/openclaw_path/11_final_architecture_map.svg)


## Финальная схема

```text
VK
→ channel bridge
→ LangGraph server
→ Deep Agent
├── _backend: filesystem / shell boundary
├── JENKINS_TOOLS: DevOps API boundary
├── VK_TOOLS: outbound messenger boundary
└── SWE Loop: task → edit → test → review
→ result
```

Финальный graph: `openclaw_05_swe`.  
VK bridge остаётся отдельным channel worker: `scripts/vk_langgraph_bridge.py`.


## Эволюция stages

| Stage | Новая способность |
| --- | --- |
| 01 Core | Deep Agent + LangGraph state |
| 02 Workspace | Filesystem backend и workspace instructions |
| 03 Jenkins | DevOps capabilities через Python tools |
| 04 VK Bridge | Messenger channel и `peer_id → thread_id` mapping |
| 05 SWE Loop | Задача → PR-like change → проверка |


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd() / 'workshop_notebooks' / 'openclaw_path'):
    if (candidate / 'workshop_utils.py').exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from workshop_utils import read_config

config = read_config()
print("Graphs:")
for name, target in sorted(config.get("graphs", {}).items()):
    print(f"- {name}: {target}")
print("Bridge: scripts/vk_langgraph_bridge.py")


## Примитивы

| Примитив | Где реализован |
| --- | --- |
| Agent loop | Deep Agents |
| State и sessions | LangGraph |
| Workspace boundary | `_backend`: FilesystemBackend / LocalShellBackend |
| DevOps actions | `JENKINS_TOOLS` |
| Messenger outbound | `VK_TOOLS` |
| Messenger inbound | `scripts/vk_langgraph_bridge.py` |
| Channel sessions | `peer_id → thread_id` mapping |
| SWE validation | Tests + reviewer/subagent prompts |


## Что есть / чего нет

Есть:

- настоящий messenger-first агент;
- реальные внешние actions;
- session continuity;
- tool traces;
- делегирование;
- SWE workflow.

Нет:

- полноценного marketplace skills;
- универсального gateway;
- production multi-user isolation;
- scheduler;
- развитой memory subsystem.


## Проверка в LangGraph Studio

### Запрос

```text
Из VK: Запусти smoke build и сообщи сюда, принял ли Jenkins запуск.
```

### Ожидаемое поведение

1. VK message приходит через bridge.
2. LangGraph запускает финальный assistant.
3. Агент запускает Jenkins smoke build.
4. Ответ с подтверждением/queue_url возвращается в VK.
5. В trace видны channel context, tool call и финальный ответ.

### Что изменилось относительно предыдущего этапа

Все слои собраны в одну систему: channel, bridge, runtime, tools, subagents, validation.

### Текущее ограничение

Это минимальный воспроизводимый runtime, а не второй OpenClaw.
